# Testing Embedding Models
---

This notebook accompanies the video tutorial on testing and using embedding models with Hugging Face and Mistral AI as part of the RAG (Retrieval-Augmented Generation) project for Willow Creek City Hall.

### Goal: Explore how to generate vector representations (embeddings) of texts to improve the retrieval of relevant information.

#### Tools used:

*   Hugging Face: To access open source embedding models through the sentence-transformers library.
*   Mistral AI: To use high-performing embedding models through their serverless API.

#### Objectives:
By comparing the similarity between the query_vector and the available document_vectors, the RAG system can identify the most relevant document to answer the question, even if the exact words are not identical.


In [ ]:
# @title Installing the libraries
!pip install sentence-transformers numpy -q
print("Libraries installed!")

In [ ]:
# @title Loading the model and vectorizing texts
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the model from the Hugging Face Hub
print("Loading the Sentence Transformer model...")
model_hf = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Model loaded!")

# Example texts (simulating a question and a document excerpt)
texts_hf = [
    "What are the City Hall opening hours?", # Simulated resident question
    "Willow Creek City Hall is open Monday to Friday from 8:30 AM to 5:00 PM.", # Simulated document info
    "The weekly market takes place every Saturday morning on the central square." # Irrelevant info
]

# Generate the embeddings
print("\nGenerating the embeddings...")
embeddings_hf = model_hf.encode(texts_hf)

print("Embeddings generated!")
print("Shape of the first embedding:", embeddings_hf[0].shape) # Shows the dimension of the vector
# Optional: Display the embeddings (can be very long)
# print("\nEmbeddings:")
# print(embeddings_hf)

In [ ]:
# @title Computing the cosine similarity

def cosine_similarity(vec1, vec2):
  """Computes the cosine similarity between two vectors."""
  dot_product = np.dot(vec1, vec2)
  norm_vec1 = np.linalg.norm(vec1)
  norm_vec2 = np.linalg.norm(vec2)
  if norm_vec1 == 0 or norm_vec2 == 0:
    return 0 # Avoid division by zero
  return dot_product / (norm_vec1 * norm_vec2)

# Embedding of the question
embedding_question = embeddings_hf[0]

# Embeddings of the potential documents
embedding_doc1 = embeddings_hf[1]
embedding_doc2 = embeddings_hf[2]

# Compute the similarities
similarity_q_doc1 = cosine_similarity(embedding_question, embedding_doc1)
similarity_q_doc2 = cosine_similarity(embedding_question, embedding_doc2)

print(f"Question: '{texts_hf[0]}'")
print("-" * 30)
print(f"Document 1: '{texts_hf[1]}'")
print(f"Similarity with the question: {similarity_q_doc1:.4f}")
print("-" * 30)
print(f"Document 2: '{texts_hf[2]}'")
print(f"Similarity with the question: {similarity_q_doc2:.4f}")
print("-" * 30)

if similarity_q_doc1 > similarity_q_doc2:
    print("\nConclusion: Document 1 is semantically closer to the question.")
else:
    print("\nConclusion: Document 2 is semantically closer to the question.")

In [ ]:
# @title Installing the Mistral AI library
!pip install mistralai
print("Mistral AI library installed!")

In [ ]:
# import os
# from mistralai import Mistral

# # Read your API key from an environment variable (never hard-code a secret)
# api_key = os.environ.get("MISTRAL_API_KEY", "YOUR_MISTRAL_API_KEY")

# # Initialize the client with version 1.6.0
# client = Mistral(api_key=api_key)
# print("Mistral client initialized.")

# # Choose the model and define the messages
# model = "mistral-large-latest"  # You can adapt the model to your needs
# messages = [
#     {"role": "user", "content": "What is the best French cheese?"}
# ]

# # Non-streaming call of the chat function via the complete method
# chat_response = client.chat.complete(
#     model=model,
#     messages=messages,
# )

# # Display the answer generated by the model
# print("Model answer:", chat_response.choices[0].message.content)

In [ ]:
import os
import numpy as np
from mistralai import Mistral
from mistralai.models import EmbeddingRequest
from sklearn.metrics.pairwise import cosine_similarity


# Read the API key from an environment variable (never hard-code a secret)
api_key = os.environ.get("MISTRAL_API_KEY", "YOUR_MISTRAL_API_KEY")

# Initialize the client
client = Mistral(api_key=api_key)
print("Mistral client initialized.")

# Texts for which we generate embeddings
texts_mistral = [
    "What are the City Hall opening hours?",
    "Willow Creek City Hall is open Monday to Friday from 8:30 AM to 5:00 PM.",
    "The weekly market takes place every Saturday morning on the central square."
]

# Mistral embedding model
model_mistral = "mistral-embed"


print(f"\nGenerating the embeddings with the model '{model_mistral}'...")

# Call the API via the create method
response = client.embeddings.create(
    model=model_mistral,
    inputs=texts_mistral
)

# Extract the embeddings from the response
embeddings_mistral_raw = response.data
# Convert to a numpy array to make the computations easier (each object has an .embedding attribute)
embeddings_mistral = np.array([item.embedding for item in embeddings_mistral_raw])

print("Mistral embeddings generated!")
print("Shape of the first Mistral embedding:", embeddings_mistral[0].shape)

# --- Similarity computation ---
print("\n--- Similarity computation (Mistral) ---")
embedding_question_mistral = embeddings_mistral[0]
embedding_doc1_mistral = embeddings_mistral[1]
embedding_doc2_mistral = embeddings_mistral[2]

similarity_q_doc1_mistral = cosine_similarity([embedding_question_mistral], [embedding_doc1_mistral])[0][0]
similarity_q_doc2_mistral = cosine_similarity([embedding_question_mistral], [embedding_doc2_mistral])[0][0]

print(f"Question: '{texts_mistral[0]}'")
print("-" * 30)
print(f"Document 1: '{texts_mistral[1]}'")
print(f"Similarity with the question: {similarity_q_doc1_mistral:.4f}")
print("-" * 30)
print(f"Document 2: '{texts_mistral[2]}'")
print(f"Similarity with the question: {similarity_q_doc2_mistral:.4f}")
print("-" * 30)

if similarity_q_doc1_mistral > similarity_q_doc2_mistral:
    print("\nConclusion (Mistral): Document 1 is semantically closer to the question.")
else:
    print("\nConclusion (Mistral): Document 2 is semantically closer to the question.")


## Conclusion and Next Steps

We have seen two approaches to generate embeddings:

#### Hugging Face (sentence-transformers):

- Pros: Free (open source models), full control over the model, works offline once downloaded.

- Cons: Requires local resources (RAM, CPU/GPU), may be less performant than state-of-the-art proprietary models.

#### Mistral AI (API):
- Pros: Very high-performing models, no infrastructure to manage (serverless), easy to integrate.

- Cons: Usage-based cost (paid), requires an internet connection, dependency on the provider.
